# Passo Firme — precisão e recall por limiar de confiança

Mede o modelo final (`modelos/best.pt`, YOLO26n, 3 classes) no **conjunto de teste** (31 fotos de São Paulo) em vários limiares de confiança e dois critérios de sobreposição:

- **IoU 0,5:** a caixa prevista precisa cobrir bem a anotada (critério usual do mAP50).
- **IoU 0,3:** basta a caixa estar no lugar certo. Para o pedestre e para o app, importa saber que há uma barreira ali, não o contorno exato.

Também mede, **por foto**, quantas fotos com barreira receberam pelo menos uma detecção: é o que decide se o relato do cidadão chega com a barreira identificada ou vai para revisão humana.

Não precisa de GPU (são só 31 imagens). Use *Executar tudo*; leva uns 5 minutos.

## 1. Instalação

In [ ]:
%pip install -q "ultralytics>=8.4.0" roboflow

## 2. Dataset (versão 1 do Roboflow) e modelo final (do GitHub)

Usa o mesmo segredo `ROBOFLOW_API_KEY` dos outros notebooks (painel da chave à esquerda, com acesso liberado).

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

WORKSPACE = "pedros-workspace-md4aw"
PROJETO = "passo-firme-calcadas"
VERSAO = 1

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
dataset = rf.workspace(WORKSPACE).project(PROJETO).version(VERSAO).download("yolov8", location="/content/dataset")

In [ ]:
!wget -q -O /content/best.pt https://github.com/predowss/passo-firme-visao/raw/main/modelos/best.pt
!ls -la /content/best.pt

## 3. Previsões no teste (uma vez, com confiança mínima de 0,01)

In [ ]:
import yaml
from pathlib import Path
from ultralytics import YOLO

base = Path(dataset.location)
nomes = yaml.safe_load((base / "data.yaml").read_text())["names"]
print("Classes:", nomes)

modelo = YOLO("/content/best.pt")
casos, arquivos = [], []
for img in sorted((base / "test" / "images").glob("*")):
    r = modelo.predict(str(img), conf=0.01, imgsz=640, verbose=False)[0]
    h, w = r.orig_shape
    reais = []
    rotulo = base / "test" / "labels" / (img.stem + ".txt")
    if rotulo.exists():
        for linha in rotulo.read_text().splitlines():
            if linha.strip():
                c, xc, yc, bw, bh = map(float, linha.split()[:5])
                reais.append((int(c), (xc - bw / 2) * w, (yc - bh / 2) * h, (xc + bw / 2) * w, (yc + bh / 2) * h))
    previstas = [(int(c), float(cf), *xyxy) for xyxy, c, cf in
                 zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist())]
    casos.append((reais, previstas))
    arquivos.append(img.name)

print(f"{len(casos)} imagens, {sum(len(r) for r, _ in casos)} barreiras anotadas")

## 4. Casamento entre previsões e anotações

Cada previsão, da mais confiante para a menos, casa com a anotação livre da mesma classe que tiver a maior sobreposição (IoU) acima do mínimo. O que sobra de previsão é falso alarme; o que sobra de anotação é barreira não encontrada.

In [ ]:
def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    uniao = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / uniao if uniao > 0 else 0.0


def contar(casos, nomes, limiar, iou_min):
    """casos: lista de (reais, previstas); reais = [(classe, x1, y1, x2, y2)], previstas = [(classe, conf, x1, y1, x2, y2)].
    Casa cada previsão (da mais confiante para a menos) com a anotação livre da mesma classe de maior IoU."""
    tp = {c: 0 for c in range(len(nomes))}
    fp = dict(tp)
    fn = dict(tp)
    for reais, previstas in casos:
        prev = sorted([p for p in previstas if p[1] >= limiar], key=lambda p: -p[1])
        livres = list(range(len(reais)))
        for p in prev:
            melhor, melhor_iou = None, iou_min
            for j in livres:
                if reais[j][0] == p[0]:
                    v = iou(reais[j][1:], p[2:])
                    if v >= melhor_iou:
                        melhor, melhor_iou = j, v
            if melhor is None:
                fp[p[0]] += 1
            else:
                tp[p[0]] += 1
                livres.remove(melhor)
        for j in livres:
            fn[reais[j][0]] += 1
    linhas = []
    for c, nome in enumerate(nomes):
        linhas.append((nome, tp[c], fp[c], fn[c]))
    linhas.append(("todas", sum(tp.values()), sum(fp.values()), sum(fn.values())))
    return linhas

## 5. Tabela por limiar e por classe

In [ ]:
import pandas as pd

LIMIARES = [0.10, 0.25, 0.35, 0.50]
linhas = []
for iou_min in (0.5, 0.3):
    for limiar in LIMIARES:
        for nome, tp, fp, fn in contar(casos, nomes, limiar, iou_min):
            linhas.append({
                "IoU mínimo": iou_min, "confiança mínima": limiar, "classe": nome,
                "acertos": tp, "falsos alarmes": fp, "não encontradas": fn,
                "precisão": round(tp / (tp + fp), 2) if tp + fp else None,
                "recall": round(tp / (tp + fn), 2) if tp + fn else None,
            })
tabela = pd.DataFrame(linhas)
pd.set_option("display.max_rows", 100)
tabela

In [ ]:
# visão resumida: só a linha "todas" e obstáculo, que é o que o modelo acha bem
resumo = tabela[tabela["classe"].isin(["todas", "obstaculo"])].pivot_table(
    index=["IoU mínimo", "confiança mínima"], columns="classe", values=["precisão", "recall"])
resumo

## 6. Por foto: a foto com barreira recebeu alguma detecção?

In [ ]:
linhas_foto = []
for limiar in LIMIARES:
    com_barreira = [(reais, prev) for reais, prev in casos if reais]
    sem_barreira = [(reais, prev) for reais, prev in casos if not reais]
    achou = sum(1 for _, prev in com_barreira if any(p[1] >= limiar for p in prev))
    alarme = sum(1 for _, prev in sem_barreira if any(p[1] >= limiar for p in prev))
    linhas_foto.append({
        "confiança mínima": limiar,
        "fotos com barreira": len(com_barreira),
        "com ao menos 1 detecção": achou,
        "% das fotos com barreira": round(100 * achou / len(com_barreira)) if com_barreira else None,
        "fotos sem barreira": len(sem_barreira),
        "com alarme falso": alarme,
    })
por_foto = pd.DataFrame(linhas_foto)
por_foto

## 7. Salvar e baixar as tabelas

O arquivo `limiares.csv` baixa para o computador; mandem para atualizar o relatório.

In [ ]:
from google.colab import files

with open("/content/limiares.csv", "w", encoding="utf-8") as f:
    f.write("# por caixa\n")
    tabela.to_csv(f, index=False)
    f.write("\n# por foto\n")
    por_foto.to_csv(f, index=False)
files.download("/content/limiares.csv")